# Results Exploration

Inspect metrics and prediction outputs saved by the training notebook.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / 'artifacts').is_dir():
        PROJECT_ROOT = candidate
        break

artifacts_dir = PROJECT_ROOT / 'artifacts'
summary_path = artifacts_dir / 'training_summary.json'
predictions_path = artifacts_dir / 'predictions.csv'
label_map_path = artifacts_dir / 'label_map.csv'

print('Artifacts:', artifacts_dir)
print('Summary exists:', summary_path.exists())
print('Predictions exists:', predictions_path.exists())

In [ ]:
if summary_path.exists():
    with open(summary_path, 'r', encoding='utf-8') as handle:
        summary = json.load(handle)
    display(summary)

if predictions_path.exists() and label_map_path.exists():
    predictions = pd.read_csv(predictions_path)
    labels = pd.read_csv(label_map_path)
    id_to_class = dict(zip(labels['label_id'], labels['class']))
    predictions['true_class'] = predictions['label'].map(id_to_class)
    predictions['pred_class'] = predictions['prediction'].map(id_to_class)
    display(predictions.head())

    confusion = pd.crosstab(predictions['true_class'], predictions['pred_class'])
    plt.figure(figsize=(12, 10))
    plt.imshow(confusion.values, aspect='auto')
    plt.colorbar()
    plt.xticks(range(len(confusion.columns)), confusion.columns, rotation=90)
    plt.yticks(range(len(confusion.index)), confusion.index)
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.show()

    mistakes = predictions[predictions['label'] != predictions['prediction']].copy()
    mistakes['correct'] = False
    display(mistakes.sort_values('confidence', ascending=False).head(20))
else:
    print('Run the training notebook first to generate predictions.csv and training_summary.json.')